In [ ]:

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1. Load dataset (works from project root or notebooks/)
candidates = [
    Path("data/Addis_Ababa_city_RTA.csv"),
    Path("../data/Addis_Ababa_city_RTA.csv"),
]

DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("Cannot find Addis_Ababa_city_RTA.csv")

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

# 2. Identify target
TARGET_COL = "Number_of_casualties"

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found!")

print("Target column:", TARGET_COL)

print("=== PHASE 1.1 — STRUCTURE AND QUALITY ===")
print(f"Dataset shape: {df.shape}")
print(f"Target column: {TARGET_COL}")
print("\nColumn types:\n", df.dtypes.value_counts())

# 3. Missing values
print("\nMissing values:")
na_counts = df.isna().sum()
print(na_counts[na_counts > 0] if na_counts.sum() else "No NaN values")

# 4. Unknown values and rare categories
cat_cols = df.select_dtypes(include=["object", "category", "string"]).columns

unk_dict = {}
rare_dict = {}

for col in cat_cols:
    s = df[col].astype("string").str.strip()

    n_unk = s.str.lower().isin(
        ["unknown", "unk", "na", "n/a", "", "none", "nan", "-"]
    ).sum()

    if n_unk > 0:
        unk_dict[col] = int(n_unk)

    freq = s.value_counts(normalize=True)
    n_rare = int((freq < 0.01).sum())

    if n_rare > 0:
        rare_dict[col] = n_rare

print("\nUnknown values:", unk_dict)
print("Rare categories (<1%):", rare_dict)
print("Exact duplicates:", df.duplicated().sum())

# 5. Target distribution
print("\n=== PHASE 1.2 — TARGET DISTRIBUTION ===")

y_eda = pd.to_numeric(df[TARGET_COL], errors="coerce")

print(y_eda.describe())
print("\nTarget frequencies:")
print(y_eda.value_counts().sort_index())

print("\nRaw skewness:", round(y_eda.skew(), 4))
print(
    "Log skewness:",
    round(np.log1p(y_eda.clip(lower=0)).skew(), 4)
)

# 6. Visualizations
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(
    y_eda.dropna(),
    discrete=True,
    ax=axes[0]
)
axes[0].set_title("Original target")

sns.histplot(
    np.log1p(y_eda.dropna().clip(lower=0)),
    ax=axes[1],
    kde=True
)
axes[1].set_title("Log-transformed target: ln(1+y)")

plt.tight_layout()
plt.show()

# 7. Leakage detection
print("\n=== PHASE 1.3 — DATA LEAKAGE ===")

leakage_cols = [
    c for c in df.columns
    if c != TARGET_COL
    and any(
        k in c.lower()
        for k in ["casualty", "fitness", "severity"]
    )
]

print("Columns to exclude:", leakage_cols)

print(
    "\nPotential bias: police reporting bias in accidents "
    "recorded between 2016 and 2022."
)


In [ ]:
# FINAL QUALITY CHECKS

print("=== UNIQUE VALUES ===")
for col in df.columns:
    print(f"{col}: {df[col].nunique(dropna=True)}")


print("\n=== UNKNOWN VALUES ===")

unknown_tokens = {
    "unknown", "na", "n/a", "?", "other"
}

for col in df.select_dtypes(include=["str", "object"]).columns:
    values = df[col].astype("string").str.strip().str.lower()
    count = values.isin(unknown_tokens).sum()

    if count > 0:
        print(f"{col}: {count}")


print("\n=== DUPLICATES ===")
print("Duplicate rows:", df.duplicated().sum())


print("\n=== TARGET VALIDATION ===")
print("Target column:", TARGET_COL)
print("\nTarget value counts:")
print(df[TARGET_COL].value_counts().sort_index())

print("\nNegative target values:", (df[TARGET_COL] < 0).sum())
print("Missing target values:", df[TARGET_COL].isna().sum())
print("Minimum target:", df[TARGET_COL].min())
print("Maximum target:", df[TARGET_COL].max())